In [1]:
import pandas as pd
from string import Template
from pathlib import Path
import json

DATA_FILE_GEMMA = Path("data/gemma-multilingual-zero-prompts.csv")
DATA_FILE_QWEN = Path("data/qwen-multilingual-zero-prompts.csv")


In [2]:
qwen_df = pd.read_csv(DATA_FILE_QWEN)
gemma_df = pd.read_csv(DATA_FILE_GEMMA)

In [3]:
marathi_df = gemma_df[gemma_df['language']=="Marathi"].copy().reset_index(drop=True)
sinhala_df = gemma_df[gemma_df['language']=="Sinhala"].copy().reset_index(drop=True)
tamil_df = gemma_df[gemma_df['language']=="Tamil"].copy().reset_index(drop=True)

In [4]:
albanian_df = qwen_df[qwen_df['language']=="Albanian"].copy().reset_index(drop=True)
odia_df = qwen_df[qwen_df['language']=="Odia"].copy().reset_index(drop=True)
hindi_df = qwen_df[qwen_df['language']=="Hindi"].copy().reset_index(drop=True)
punjabi_df = qwen_df[qwen_df['language']=="Punjabi"].copy().reset_index(drop=True)

In [6]:
# Read the Excel file
examples_df = pd.read_excel(
    "data/albania.xlsx"
)

In [7]:
examples_df

,Curriculum,Grade,Topic,Learning objectives,Questions,Translated Questions,Answer,Translated answer
0,ANC ( Albanian National Curriculum (Kurrikula ...,Class 3,Number and Operations,"Calculate the sum of two 2-digit numbers, corr...",Ana ka 27 mollë dhe blen edhe 16 të tjera në t...,Ana has 27 apples and buys 16 more at the mark...,43 mollë,43 apples
1,ANC ( Albanian National Curriculum (Kurrikula ...,Class 3,Number and Operations,Calculate the difference of two 2-digit number...,"Ina kishte 27 mollë. Ajo i dha shoqes së saj, ...",Ina had 27 apples. She gave 9 apples to her fr...,18 mollë,18 apples
2,ANC ( Albanian National Curriculum (Kurrikula ...,Class 3,Number and Operations,Multiply and add small numbers in practical co...,Arba ka 2 monedha me vlerë 50 lekë dhe 3 moned...,Arba has 2 coins worth 50 lek each and 3 coins...,160 lekë,160 lek
3,ANC ( Albanian National Curriculum (Kurrikula ...,Class 3,Number and Operations,Use division of whole numbers to solve problem...,Një librari shet libra për 2 lekë secilin. Nës...,A bookstore sells books for 2 lek each. If Arl...,4 libra,4 books
4,ANC ( Albanian National Curriculum (Kurrikula ...,Class 3,Number and Operations,Calculate a given percentage of a number and d...,Një klasë ka 40 nxënës dhe 10 prej tyre kanë ç...,If there are 40 students in a class and 10 of ...,25%,25%
5,ANC ( Albanian National Curriculum (Kurrikula ...,Class 3,Operations,Translate word problems involving one or two a...,Një librari shiti gjithsej 48 libra gjatë një ...,A bookstore sold 48 books in one day. The numb...,16 libra,16 books
6,ANC ( Albanian National Curriculum (Kurrikula ...,Class 3,Operations,"Solve practical word problems involving money,...",Një librari në Tiranë shet një libër për 85 le...,A bookstore in Tirana sells a notebook for 85 ...,60 lekë,60 lek
7,ANC ( Albanian National Curriculum (Kurrikula ...,Class 3,Measurement,Measure the length of an object to the nearest...,Një autobus lokal përshkon një linjë 10 km të ...,A local bus travels a route that is 10 km long...,80 kilometers,80 kilometers
8,ANC ( Albanian National Curriculum (Kurrikula ...,Class 3,Measurement,Measure the mass of an object using grams (g) ...,Një kompani në Dajt grumbullon 1 250 kg pleh. ...,A company in Dajt collects 95 kg of fertilizer...,53 kg,53 kg
9,ANC ( Albanian National Curriculum (Kurrikula ...,Class 3,Data handeling,Solve one-step addition problems involving rea...,"Në një teatër kanë ardhur 23 fëmijë, dhe më vo...","At a theater, 23 children arrive, and later 17...",40 fëmijë,40 children


In [8]:

# --------------------------------------------------
# Create numeric grade column
# "Grade 3" -> 3
# "Grade 4" -> 4
# --------------------------------------------------
examples_df["grade"] = (
    examples_df["Grade"]
    .str.extract(r"(\d+)")
    .astype(int)
)

# --------------------------------------------------
# Create topic numbering that resets within each grade
# --------------------------------------------------
examples_df["topic_num"] = (
    examples_df.groupby("grade")
    .cumcount()
    .add(1)
)

# --------------------------------------------------
# Create topic column
# Format:
# 1. Topic : Learning Objectives
# 2. Topic : Learning Objectives
# ...
# --------------------------------------------------
examples_df["topic"] = (
    examples_df["topic_num"].astype(str)
    + ". "
    + examples_df["Topic"].astype(str).str.strip()
    + " : "
    + examples_df["Learning objectives"].astype(str).str.strip()
)

# Optional cleanup
examples_df = examples_df.drop(columns=["topic_num"])

In [10]:

# Create example_1 column
examples_df["example_1"] = examples_df.apply(
    lambda row: json.dumps(
        {
            "question": row["Questions"],
            "answer": row["Answer"]
        },
        ensure_ascii=False
    ) if pd.notna(row["Questions"]) and pd.notna(row["Answer"]) else None,
    axis=1
)

# Optional: remove the temporary columns


In [18]:
examples_df["country"] = "Albania"
examples_df["curriculum"] = "Albanian National"
examples_df["addressing"] = "Albanian"
examples_df["language"] = "Albanian"

In [24]:
examples_df["topic_list"] = (
    examples_df
    .groupby("grade")["topic"]
    .transform(lambda topics: "\n".join(topics.astype(str)))
)

In [39]:
examples_df = examples_df.drop(columns=["Curriculum", "Grade", "Topic", "Learning objectives", "Questions", "Translated Questions", "Answer", "Translated answer"])

In [41]:
examples_df.to_excel(
    "data/albanian.xlsx",
    index=False
)

In [8]:
examples = pd.read_excel(
    "data/india.xlsx",
    sheet_name="Correct_3ver_Final_Updated"
)

examples = examples.rename(columns={"Grade": "grade"})

examples = examples.reset_index(drop=True)

In [9]:


# Create one row per unique grade-topic-learning objective
unique_topics = (
    examples[["grade", "Topic", "Learning Objective"]]
    .drop_duplicates()
    .copy()
)

# Number unique topics within each grade
unique_topics["topic_num"] = (
    unique_topics.groupby("grade")
    .cumcount()
    .add(1)
)

# Merge topic numbers back to all example rows
examples = examples.merge(
    unique_topics,
    on=["grade", "Topic", "Learning Objective"],
    how="left"
)

# Create topic column
examples["topic"] = (
    examples["topic_num"].astype(str)
    + ". "
    + examples["Topic"].astype(str).str.strip()
    + " : "
    + examples["Learning Objective"].astype(str).str.strip()
)

examples = examples.drop(columns=["topic_num"])

In [10]:


# Make sure matching columns have same names/types
examples["grade"] = examples["grade"].astype(int)
hindi_df["grade"] = hindi_df["grade"].astype(int)

examples["topic"] = examples["topic"].astype(str).str.strip()
hindi_df["topic"] = hindi_df["topic"].astype(str).str.strip()

# Create JSON string for each example row
examples["example_json"] = examples.apply(
    lambda row: json.dumps(
        {
            "question": row["Hindi question"],
            "answer": row["Answer_Hindi"]
        },
        ensure_ascii=False
    ),
    axis=1
)



# Number examples within each grade-topic pair
examples["example_num"] = (
    examples.groupby(["grade", "topic"])
    .cumcount()
    .add(1)
)

# Keep only the first 10 examples for each grade-topic pair
examples = examples[examples["example_num"] <= 10].copy()

# Convert example_num to column names
examples["example_col"] = "example_" + examples["example_num"].astype(str)

# Pivot into wide format
examples_wide = examples.pivot_table(
    index=["grade", "topic"],
    columns="example_col",
    values="example_json",
    aggfunc="first"
).reset_index()

examples_wide.columns.name = None



In [24]:
hindi_meta = hindi_df[
    ["country", "curriculum", "addressing", "grade", "topic_list", "language"]
].drop_duplicates(subset=["grade"])

In [26]:
hindi = examples_wide.merge(
    hindi_meta,
    on=["grade"],
    how="left"
)

In [27]:
hindi

,grade,topic,example_1,example_10,example_2,example_3,example_4,example_5,example_6,example_7,example_8,example_9,country,curriculum,addressing,topic_list,language
0,3,1. Counting : reads and writes numbers up to 9...,"{""question"": ""गौरव का रोल नंबर 150 है। गौरव के...","{""question"": ""संख्या 572 में, सैकड़ों के स्थान...","{""question"": ""संख्या 572 में, सैकड़ों के स्थान...","{""question"": ""राहुल का रोल नंबर 299 के बाद है।...","{""question"": ""संख्या 719 में, कौन सा अंक दशकों...","{""question"": ""मैं एक 3-अंकों वाली संख्या हूँ। ...","{""question"": ""संख्या 692 में, सैकड़ों के स्थान...","{""question"": ""सबसे बड़े 3-अंकीय नंबर बनाने के ...","{""question"": ""संख्या 864 में, दस के स्थान पर क...","{""question"": ""स्नेहा का रोल नंबर 26 है। स्नेहा...",India,NCERT,Indian,1. Counting : reads and writes numbers up to 9...,Hindi
1,3,10. measurement of weight : weighs objects usi...,"{""question"": ""तरबूज अनानास से भारी होता है। अन...",NaN,"{""question"": ""एक बिल्ली खरगोश से भारी होती है।...","{""question"": ""एक किताब कापी से भारी होती है। ए...","{""question"": ""एक धातु का बर्तन प्लास्टिक की बा...","{""question"": ""एक आम केले से भारी है। एक केला आ...","{""question"": ""गेहूं की एक बोरी का वजन 50 किलोग...","{""question"": ""चावल का एक थैला 5 किलोग्राम का ह...","{""question"": ""रिया ने 250 ग्राम वज़न का बिस्कि...",NaN,India,NCERT,Indian,1. Counting : reads and writes numbers up to 9...,Hindi
2,3,11. measurement of capacity : compares the cap...,"{""question"": ""एक जग में 3 लीटर पानी आता है। एक...",NaN,"{""question"": ""एक बाल्टी में 10 लीटर पानी आता ह...","{""question"": ""रवि की बोतल में मीना की बोतल से ...","{""question"": ""एक चायदानी में कप से ज़्यादा चाय...","{""question"": ""एक स्टील के बर्तन में 6 कप पानी ...","{""question"": ""आशा एक कप से पानी डालकर अपना फूल...","{""question"": ""सीता एक मग से बाल्टी भरती है। बा...","{""question"": ""रोहन के पास एक जग है जिससे 5 गिल...",NaN,India,NCERT,Indian,1. Counting : reads and writes numbers up to 9...,Hindi
3,3,12. Calendar : identifies a particular day and...,"{""question"": ""आज 10 जनवरी है। भारत में गणतंत्र...","{""question"": ""रिया ने 15 मई को अपनी गर्मियों क...","{""question"": ""दिल्ली में एक स्कूल 15 मई को गर्...","{""question"": ""अमित का जन्मदिन 02/04/2015 को है...","{""question"": ""अनिल का जन्मदिन 6/7/2015 को है। ...","{""question"": ""रिया का स्कूल दिवाली के लिए 1 नव...","{""question"": ""एक विज्ञान प्रदर्शनी 5 जनवरी से ...","{""question"": ""इस साल रक्षाबंधन 19 अगस्त को है।...","{""question"": ""एक क्रिकेट टूर्नामेंट 10 दिसंबर ...","{""question"": ""रिया की अंतिम परीक्षाएं 4 मार्च ...",India,NCERT,Indian,1. Counting : reads and writes numbers up to 9...,Hindi
4,3,13. time : reads the time correctly to the hou...,"{""question"": ""राहुल ने शाम को 4:30 बजे अपना अभ...","{""question"": ""मीना ने पूर्वाह्न 10:20 बजे पेंट...","{""question"": ""एक फुटबॉल मैच अपराह्न 3:20 बजे श...","{""question"": ""एक केक को अपराह्न 2:10 बजे ओवन म...","{""question"": ""स्कूल की प्रार्थना सभा पूर्वाह्न...","{""question"": ""कुणाल ने अपराह्न 6:10 बजे खेलना ...","{""question"": ""अर्जुन पूर्वाह्न 7:45 बजे स्कूल ...","{""question"": ""नेहा ने अपराह्न 3:15 बजे खेलना श...","{""question"": ""रमेश ने अपराह्न 2:10 बजे किताब प...","{""question"": ""सीता ने अपराह्न 5:30 बजे खाना बन...",India,NCERT,Indian,1. Counting : reads and writes numbers up to 9...,Hindi
5,3,14. Pattern : extends patterns in numbers,"{""question"": ""रिया आमों को एक पैटर्न में सजा र...","{""question"": ""अनुपस्थित संख्या भरें:\n90, 80, ...","{""question"": ""योग करने वाले छात्रों की संख्या ...","{""question"": ""एक श्रेणी 9 का पहाड़ा दिखाती है:...","{""question"": ""एक बिंदु पैटर्न इस तरह बढ़ता है:...","{""question"": ""एक रेलवे कोच नंबर का पैटर्न इस प...","{""question"": ""रानी लड्डूओं को एक पैटर्न में सज...","{""question"": ""मोहन पूजा के लिए फूल लगा रहा है:...","{""question"": ""एक पैटर्न हर बार 5 से घटता है: ...","{""question"": ""डोमिनो शैली का डॉट पैटर्न बढ़ता ...",India,NCERT,Indian,1. Counting : reads and writes 

In [29]:
hindi.to_excel(
    "data/hindi.xlsx",
    index=False
)

In [30]:
examples = pd.read_excel(
    "data/india.xlsx",
    sheet_name="Correct_3ver_Final_Updated"
)

examples = examples.rename(columns={"Grade": "grade"})

examples = examples.reset_index(drop=True)

In [31]:


# Create one row per unique grade-topic-learning objective
unique_topics = (
    examples[["grade", "Topic", "Learning Objective"]]
    .drop_duplicates()
    .copy()
)

# Number unique topics within each grade
unique_topics["topic_num"] = (
    unique_topics.groupby("grade")
    .cumcount()
    .add(1)
)

# Merge topic numbers back to all example rows
examples = examples.merge(
    unique_topics,
    on=["grade", "Topic", "Learning Objective"],
    how="left"
)

# Create topic column
examples["topic"] = (
    examples["topic_num"].astype(str)
    + ". "
    + examples["Topic"].astype(str).str.strip()
    + " : "
    + examples["Learning Objective"].astype(str).str.strip()
)

examples = examples.drop(columns=["topic_num"])

In [32]:


# Make sure matching columns have same names/types
examples["grade"] = examples["grade"].astype(int)
marathi_df["grade"] = marathi_df["grade"].astype(int)

examples["topic"] = examples["topic"].astype(str).str.strip()
marathi_df["topic"] = marathi_df["topic"].astype(str).str.strip()

# Create JSON string for each example row
examples["example_json"] = examples.apply(
    lambda row: json.dumps(
        {
            "question": row["Marathi Question"],
            "answer": row["Answer Marathi"]
        },
        ensure_ascii=False
    ),
    axis=1
)



# Number examples within each grade-topic pair
examples["example_num"] = (
    examples.groupby(["grade", "topic"])
    .cumcount()
    .add(1)
)

# Keep only the first 10 examples for each grade-topic pair
examples = examples[examples["example_num"] <= 10].copy()

# Convert example_num to column names
examples["example_col"] = "example_" + examples["example_num"].astype(str)

# Pivot into wide format
examples_wide = examples.pivot_table(
    index=["grade", "topic"],
    columns="example_col",
    values="example_json",
    aggfunc="first"
).reset_index()

examples_wide.columns.name = None



In [33]:
marathi_meta = marathi_df[
    ["country", "curriculum", "addressing", "grade", "topic_list", "language"]
].drop_duplicates(subset=["grade"])

marathi = examples_wide.merge(
    marathi_meta,
    on=["grade"],
    how="left"
)

In [35]:
marathi.to_excel(
    "data/marathi.xlsx",
    index=False
)

In [39]:
examples = pd.read_excel(
    "data/india.xlsx",
    sheet_name="Correct_3ver_Final_Updated"
)

examples = examples.rename(columns={"Grade": "grade"})

examples = examples.reset_index(drop=True)

In [40]:


# Create one row per unique grade-topic-learning objective
unique_topics = (
    examples[["grade", "Topic", "Learning Objective"]]
    .drop_duplicates()
    .copy()
)

# Number unique topics within each grade
unique_topics["topic_num"] = (
    unique_topics.groupby("grade")
    .cumcount()
    .add(1)
)

# Merge topic numbers back to all example rows
examples = examples.merge(
    unique_topics,
    on=["grade", "Topic", "Learning Objective"],
    how="left"
)

# Create topic column
examples["topic"] = (
    examples["topic_num"].astype(str)
    + ". "
    + examples["Topic"].astype(str).str.strip()
    + " : "
    + examples["Learning Objective"].astype(str).str.strip()
)

examples = examples.drop(columns=["topic_num"])

In [41]:


# Make sure matching columns have same names/types
examples["grade"] = examples["grade"].astype(int)
odia_df["grade"] = odia_df["grade"].astype(int)

examples["topic"] = examples["topic"].astype(str).str.strip()
odia_df["topic"] = odia_df["topic"].astype(str).str.strip()

# Create JSON string for each example row
examples["example_json"] = examples.apply(
    lambda row: json.dumps(
        {
            "question": row["Odia question"],
            "answer": row["Answer_Odia"]
        },
        ensure_ascii=False
    ),
    axis=1
)



# Number examples within each grade-topic pair
examples["example_num"] = (
    examples.groupby(["grade", "topic"])
    .cumcount()
    .add(1)
)

# Keep only the first 10 examples for each grade-topic pair
examples = examples[examples["example_num"] <= 10].copy()

# Convert example_num to column names
examples["example_col"] = "example_" + examples["example_num"].astype(str)

# Pivot into wide format
examples_wide = examples.pivot_table(
    index=["grade", "topic"],
    columns="example_col",
    values="example_json",
    aggfunc="first"
).reset_index()

examples_wide.columns.name = None


In [42]:
odia_meta = odia_df[
    ["country", "curriculum", "addressing", "grade", "topic_list", "language"]
].drop_duplicates(subset=["grade"])

odia = examples_wide.merge(
    odia_meta,
    on=["grade"],
    how="left"
)

In [44]:
odia.to_excel(
    "data/odia.xlsx",
    index=False
)

In [45]:
examples = pd.read_excel(
    "data/india.xlsx",
    sheet_name="Correct_3ver_Final_Updated"
)

examples = examples.rename(columns={"Grade": "grade"})

examples = examples.reset_index(drop=True)

In [46]:


# Create one row per unique grade-topic-learning objective
unique_topics = (
    examples[["grade", "Topic", "Learning Objective"]]
    .drop_duplicates()
    .copy()
)

# Number unique topics within each grade
unique_topics["topic_num"] = (
    unique_topics.groupby("grade")
    .cumcount()
    .add(1)
)

# Merge topic numbers back to all example rows
examples = examples.merge(
    unique_topics,
    on=["grade", "Topic", "Learning Objective"],
    how="left"
)

# Create topic column
examples["topic"] = (
    examples["topic_num"].astype(str)
    + ". "
    + examples["Topic"].astype(str).str.strip()
    + " : "
    + examples["Learning Objective"].astype(str).str.strip()
)

examples = examples.drop(columns=["topic_num"])

In [47]:


# Make sure matching columns have same names/types
examples["grade"] = examples["grade"].astype(int)
punjabi_df["grade"] = punjabi_df["grade"].astype(int)

examples["topic"] = examples["topic"].astype(str).str.strip()
punjabi_df["topic"] = punjabi_df["topic"].astype(str).str.strip()

# Create JSON string for each example row
examples["example_json"] = examples.apply(
    lambda row: json.dumps(
        {
            "question": row["punjabi question"],
            "answer": row["Answer_punjabi"]
        },
        ensure_ascii=False
    ),
    axis=1
)



# Number examples within each grade-topic pair
examples["example_num"] = (
    examples.groupby(["grade", "topic"])
    .cumcount()
    .add(1)
)

# Keep only the first 10 examples for each grade-topic pair
examples = examples[examples["example_num"] <= 10].copy()

# Convert example_num to column names
examples["example_col"] = "example_" + examples["example_num"].astype(str)

# Pivot into wide format
examples_wide = examples.pivot_table(
    index=["grade", "topic"],
    columns="example_col",
    values="example_json",
    aggfunc="first"
).reset_index()

examples_wide.columns.name = None


In [48]:
punjabi_meta = punjabi_df[
    ["country", "curriculum", "addressing", "grade", "topic_list", "language"]
].drop_duplicates(subset=["grade"])

punjabi = examples_wide.merge(
    punjabi_meta,
    on=["grade"],
    how="left"
)

In [50]:
punjabi.to_excel(
    "data/punjabi.xlsx",
    index=False
)

In [8]:
examples = pd.read_excel(
    "data/lanka.xlsx",
    sheet_name="Sri Lankan - Final"
)
examples = examples.rename(columns={"topic": "Topic"})

examples = examples.reset_index(drop=True)

In [10]:


# Create one row per unique grade-topic-learning objective
unique_topics = (
    examples[["grade", "Topic", "learning_objective"]]
    .drop_duplicates()
    .copy()
)

examples["selected_question_sinhala"] = (
    examples["question_sinhala_corrected"]
    .fillna(examples["question_sinhala"])
)


# Number unique topics within each grade
unique_topics["topic_num"] = (
    unique_topics.groupby("grade")
    .cumcount()
    .add(1)
)

# Merge topic numbers back to all example rows
examples = examples.merge(
    unique_topics,
    on=["grade", "Topic", "learning_objective"],
    how="left"
)

# Create topic column
examples["topic"] = (
    examples["topic_num"].astype(str)
    + ". "
    + examples["Topic"].astype(str).str.strip()
    + " : "
    + examples["learning_objective"].astype(str).str.strip()
)

examples = examples.drop(columns=["topic_num"])

In [12]:


# Make sure matching columns have same names/types
examples["grade"] = examples["grade"].astype(int)
sinhala_df["grade"] = sinhala_df["grade"].astype(int)

examples["topic"] = examples["topic"].astype(str).str.strip()
sinhala_df["topic"] = sinhala_df["topic"].astype(str).str.strip()

# Create JSON string for each example row
examples["example_json"] = examples.apply(
    lambda row: json.dumps(
        {
            "question": row["selected_question_sinhala"],
            "answer": row["answer_sinhala"]
        },
        ensure_ascii=False
    ),
    axis=1
)



# Number examples within each grade-topic pair
examples["example_num"] = (
    examples.groupby(["grade", "topic"])
    .cumcount()
    .add(1)
)

# Keep only the first 10 examples for each grade-topic pair
examples = examples[examples["example_num"] <= 10].copy()

# Convert example_num to column names
examples["example_col"] = "example_" + examples["example_num"].astype(str)

# Pivot into wide format
examples_wide = examples.pivot_table(
    index=["grade", "topic"],
    columns="example_col",
    values="example_json",
    aggfunc="first"
).reset_index()

examples_wide.columns.name = None

sinhala_df = sinhala_df.merge(
    examples_wide,
    on=["grade", "topic"],
    how="left"
)

TypeError: Object of type datetime is not JSON serializable